In [1]:
!pip install -q transformers datasets accelerate soundfile librosa jiwer evaluate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 40.8 MB/s eta 0:00:00


In [2]:
import torch
import re
import json
import os
import numpy as np
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Union

from datasets import load_from_disk, Audio
from transformers import (
    Wav2Vec2CTCTokenizer,
    Wav2Vec2FeatureExtractor,
    Wav2Vec2Processor,
    Wav2Vec2ForCTC,
    TrainingArguments,
    Trainer,
)
from peft import LoraConfig, get_peft_model
import evaluate

print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

GPU available: True
GPU: Tesla T4


In [3]:
# Set the correct absolute paths to your Kaggle dataset
train_path = "/kaggle/input/datasets/aadityanair1/mls-spanish/mls_spanish_train"
test_path  = "/kaggle/input/datasets/aadityanair1/mls-spanish/mls_spanish_test"

train_dataset = load_from_disk(train_path)
test_dataset  = load_from_disk(test_path)

# Ensure sampling rate is 16 kHz
train_dataset = train_dataset.cast_column("audio", Audio(sampling_rate=16_000))
test_dataset  = test_dataset.cast_column("audio",  Audio(sampling_rate=16_000))

print(f"Train columns : {train_dataset.column_names}")
print(f"Train size    : {len(train_dataset)}")
print(f"Test size     : {len(test_dataset)}")

Train columns : ['audio', 'original_path', 'begin_time', 'end_time', 'transcript', 'audio_duration', 'speaker_id', 'chapter_id', 'file', 'id']
Train size    : 9000
Test size     : 1000


In [4]:
chars_to_ignore_regex = r'[\,\.\!\-\;\:\"\n¿¡]'

def extract_all_chars(batch):
    # Find the text column (MLS uses 'transcript'; FLEURS uses 'sentence')
    text_col = next(
        (c for c in ("transcript", "sentence", "text") if c in batch),
        None
    )
    if text_col is None:
        raise ValueError(f"No text column found. Available: {list(batch.keys())}")

    all_text = " ".join(
        re.sub(chars_to_ignore_regex, '', t).lower()
        for t in batch[text_col]
    )
    vocab = list(set(all_text))
    return {"vocab": [vocab], "all_text": [all_text]}

vocabs_train = train_dataset.map(
    extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=train_dataset.column_names
)
vocabs_test = test_dataset.map(
    extract_all_chars, batched=True, batch_size=-1,
    keep_in_memory=True, remove_columns=test_dataset.column_names
)

vocab_list = list(set(vocabs_train["vocab"][0]) | set(vocabs_test["vocab"][0]))
vocab_dict = {v: k for k, v in enumerate(vocab_list)}

# Replace space with pipe (word delimiter for CTC)
if " " in vocab_dict:
    vocab_dict["|"] = vocab_dict.pop(" ")

vocab_dict["[UNK]"] = len(vocab_dict)
vocab_dict["[PAD]"] = len(vocab_dict)

print(f"Total characters in vocabulary: {len(vocab_dict)}")
print(sorted(vocab_dict.keys()))

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Total characters in vocabulary: 37
["'", '[PAD]', '[UNK]', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '|', 'á', 'é', 'í', 'ñ', 'ó', 'ú', 'ü']


In [5]:
# Save vocab and build tokenizer
with open("vocab.json", "w") as f:
    json.dump(vocab_dict, f)

tokenizer = Wav2Vec2CTCTokenizer(
    "./vocab.json",
    unk_token="[UNK]",
    pad_token="[PAD]",
    word_delimiter_token="|"
)

feature_extractor = Wav2Vec2FeatureExtractor(
    feature_size=1,
    sampling_rate=16000,
    padding_value=0.0,
    do_normalize=True,
    return_attention_mask=True
)

processor = Wav2Vec2Processor(feature_extractor=feature_extractor, tokenizer=tokenizer)
processor.save_pretrained("./wav2vec2-xls-r-fleurs-spanish")

print("Processor saved.")
print(f"Vocab size: {len(processor.tokenizer)}")
print(processor.tokenizer.get_vocab())

Processor saved.
Vocab size: 39
{'s': 0, 'i': 1, 'x': 2, 'ó': 3, 'f': 4, 'z': 5, 'í': 6, 'j': 7, 'r': 8, 'ñ': 9, 'y': 10, 'u': 11, 'l': 12, 'o': 13, 'm': 14, 'a': 15, 'b': 16, 'w': 17, 'á': 18, 'g': 19, 'p': 20, 'h': 21, 'é': 22, 'n': 23, 't': 24, "'": 25, 'd': 26, 'ü': 27, 'e': 29, 'q': 30, 'c': 31, 'v': 32, 'k': 33, 'ú': 34, '|': 28, '[UNK]': 35, '[PAD]': 36, '<s>': 37, '</s>': 38}


In [6]:
def prepare_dataset_for_model(batch):
    audio = batch["audio"]

    # --- Audio ---
    batch["input_values"] = processor(
        audio["array"], sampling_rate=audio["sampling_rate"]
    ).input_values[0]

    # --- Text: find whichever column exists ---
    target_text = batch.get("transcript") or batch.get("sentence") or batch.get("text")
    if target_text is None:
        raise ValueError(f"No text column found. Keys: {list(batch.keys())}")

    clean_text = re.sub(chars_to_ignore_regex, '', target_text).lower()
    batch["labels"] = processor(text=clean_text).input_ids
    return batch

columns_to_remove = train_dataset.column_names

# Cache to a writable directory to avoid read-only errors on Kaggle
cache_dir = "/kaggle/working/dataset_cache"
os.makedirs(cache_dir, exist_ok=True)

train_dataset = train_dataset.map(
    prepare_dataset_for_model,
    remove_columns=columns_to_remove,
    batched=False,
    cache_file_name=os.path.join(cache_dir, "train_cache.arrow")
)
test_dataset = test_dataset.map(
    prepare_dataset_for_model,
    remove_columns=test_dataset.column_names,
    batched=False,
    cache_file_name=os.path.join(cache_dir, "test_cache.arrow")
)

print(f"Columns remaining: {train_dataset.column_names}")
print(f"Train size: {len(train_dataset)}")

Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Columns remaining: ['input_values', 'labels']
Train size: 9000


In [7]:
sample = train_dataset[0]
print(f"Remaining Columns : {train_dataset.column_names}")
print(f"Sample Label IDs  : {sample['labels'][:10]}")
print(f"Decoded Sample    : {processor.decode(sample['labels'])}")

Remaining Columns : ['input_values', 'labels']
Sample Label IDs  : [29, 12, 28, 26, 11, 30, 11, 29, 28, 29]
Decoded Sample    : el duque entonces volviéndose á nosotros nos mandó con imperiosas razones los dejásemos solos y que viniesemos á este lugar donde le esperásemos sin tener osadía de volver solamente el rostro


In [8]:
@dataclass
class DataCollatorCTCWithPadding:
    processor: Wav2Vec2Processor
    padding: Union[bool, str] = True

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_values": f["input_values"]} for f in features]
        label_features = [{"input_ids": f["labels"]}          for f in features]

        batch = self.processor.pad(input_features, padding=self.padding, return_tensors="pt")

        labels_batch = self.processor.pad(labels=label_features, padding=self.padding, return_tensors="pt")
        # Replace padding index with -100 so loss ignores it
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        batch["labels"] = labels
        return batch

data_collator = DataCollatorCTCWithPadding(processor=processor, padding=True)

# Quick smoke-test
features = [train_dataset[i] for i in range(2)]
batch = data_collator(features)
print(batch.keys())
print("Labels shape:", batch["labels"].shape)

KeysView({'input_values': tensor([[-0.0027, -0.0022, -0.0026,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0097, -0.0045, -0.0068,  ..., -0.0252, -0.0380, -0.0259]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1]], dtype=torch.int32), 'labels': tensor([[  29,   12,   28,   26,   11,   30,   11,   29,   28,   29,   23,   24,
           13,   23,   31,   29,    0,   28,   32,   13,   12,   32,    1,   22,
           23,   26,   13,    0,   29,   28,   18,   28,   23,   13,    0,   13,
           24,    8,   13,    0,   28,   23,   13,    0,   28,   14,   15,   23,
           26,    3,   28,   31,   13,   23,   28,    1,   14,   20,   29,    8,
            1,   13,    0,   15,    0,   28,    8,   15,    5,   13,   23,   29,
            0,   28,   12,   13,    0,   28,   26,   29,    7,   18,    0,   29,
           14,   13,    0,   28,    0,   13,   12,   13,    0,   28,   10,   28,
           30,   11,   29,   28,   32,    1,   23,    1,   29,    0

In [9]:
wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids    = np.argmax(pred_logits, axis=-1)
    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str  = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    return {"wer": wer, "cer": cer}

print("Metrics ready.")

Metrics ready.


In [10]:
model = Wav2Vec2ForCTC.from_pretrained(
    "facebook/wav2vec2-xls-r-300m",
    attention_dropout=0.0,
    hidden_dropout=0.0,
    feat_proj_dropout=0.0,
    mask_time_prob=0.05,
    layerdrop=0.0,          # CRITICAL: keep all layers during LoRA training
    ctc_loss_reduction="mean",
    pad_token_id=processor.tokenizer.pad_token_id,
    vocab_size=len(processor.tokenizer),
)

peft_config = LoraConfig(
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj"],
    inference_mode=False,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    modules_to_save=["lm_head"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/422 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

Wav2Vec2ForCTC LOAD REPORT from: facebook/wav2vec2-xls-r-300m
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
lm_head.bias                 | MISSING    | 
lm_head.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 3,185,703 || all params: 318,664,398 || trainable%: 0.9997


In [11]:
# Spec-augment settings
model.config.mask_time_prob    = 0.075
model.config.mask_feature_prob = 0.05
model.config.apply_spec_augment = True
model.config.layerdrop          = 0.0
model.config.ctc_loss_reduction = "mean"

# Fix embedding save bug for PEFT + Wav2Vec2
model.get_input_embeddings  = lambda: model.base_model.model.wav2vec2.feature_extractor
model.get_output_embeddings = lambda: model.base_model.model.lm_head

training_args = TrainingArguments(
    output_dir="./xlsr-spanish-lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    gradient_checkpointing=True,
    fp16=True,
    learning_rate=2e-4,
    num_train_epochs=5,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    logging_steps=5,
    report_to="none",
    remove_unused_columns=False,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    data_collator=data_collator,
    args=training_args,
    compute_metrics=compute_metrics,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=processor,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 38, 'bos_token_id': 37}.
/usr/local/lib/python3.12/dist-packages/torch/backends/cudnn/__init__.py:145: UserWarning: Please use the new API settings to control TF32 behavior, such as torch.backends.cudnn.conv.fp32_precision = 'tf32' or torch.backends.cuda.matmul.fp32_precision = 'ieee'. Old settings, e.g, torch.backends.cuda.matmul.allow_tf32 = True, torch.backends.cudnn.allow_tf32 = True, allowTF32CuDNN() and allowTF32CuBLAS() will be deprecated after Pytorch 2.9. Please see https://pytorch.org/docs/main/notes/cuda.html#tensorfloat-32-tf32-on-ampere-and-later-devices (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:80.)
  torch._C._get_cudnn_allow_tf32(),
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarnin

Epoch,Training Loss,Validation Loss,Wer,Cer
1,92.165979,5.761371,0.999885,0.995025
2,87.417181,5.503300,0.999943,0.966037
3,31.086749,1.920303,0.635098,0.205234
4,24.010864,1.529577,0.527696,0.166365
5,22.069653,1.504881,0.527954,0.167507


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embeddi

TrainOutput(global_step=1410, training_loss=66.89320497986273, metrics={'train_runtime': 31998.0578, 'train_samples_per_second': 1.406, 'train_steps_per_second': 0.044, 'total_flos': 2.2980844968926794e+19, 'train_loss': 66.89320497986273, 'epoch': 5.0})

In [12]:
trainer.save_model("./final-spanish-model")
processor.save_pretrained("./final-spanish-model")
print("Model and processor saved to ./final-spanish-model")

Model and processor saved to ./final-spanish-model


/usr/local/lib/python3.12/dist-packages/peft/utils/save_and_load.py:309: UserWarning: Setting `save_embedding_layers` to `True` as the embedding layer has been resized during finetuning.
  warnings.warn(
